In [0]:
STORAGE_ACCOUNT_KEY = dbutils.secrets.get(scope ="retailpulse-scope", key= "retailpulse-account-key")
STORAGE_ACCOUNT_NAME = "retailpulsestorage" 
CONTAINER_NAME       = "raw-data"

In [0]:
spark.conf.set(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.blob.core.windows.net",
    STORAGE_ACCOUNT_KEY
)

In [0]:
mount_path = dbutils.fs.ls(f"wasbs://{CONTAINER_NAME}@{STORAGE_ACCOUNT_NAME}.blob.core.windows.net")

display(mount_path)

In [0]:
file_path = f"{mount_path[0].path}"

In [0]:
df_raw = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)


print(f"Total rows: {df_raw.count():,}")
print(f"Total columns: {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
display(df_raw.limit(20))

In [0]:
#Null value analysis 
from pyspark.sql.functions import col, count, when, round as spark_round

total_rows = df_raw.count()

null_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_raw.columns
])

print(f"Total rows: {total_rows:,}\n")
print("Null counts per column:")
display(null_counts)

In [0]:

from pyspark.sql.functions import col, count, when

null_customer_count = df_raw.filter(col("Customer ID").isNull()).count()
total = df_raw.count()
pct = (null_customer_count / total) * 100

print(f"Null Customer IDs: {null_customer_count:,} ({pct:.1f}% of total rows)")

In [0]:
#cancelled orders 
from pyspark.sql.functions import col

df_cancelled = df_raw.filter(col("Invoice").startswith("C"))
df_normal = df_raw.filter(~col("Invoice").startswith("C"))

cancelled_pct = (df_cancelled.count()/total_rows) * 100

print(f"Normal orders: {df_normal.count():,}")
print(f"Cancelled orders: {df_cancelled.count():,} ({cancelled_pct:.1f}%)")

display(df_cancelled.limit(5))

In [0]:
#invalid quantities and prices 
df_neg_qty   = df_raw.filter(col("Quantity") <= 0)
df_zero_price = df_raw.filter(col("Price") <= 0)
df_neg_price  = df_raw.filter(col("Price") < 0)

print(f"Rows with Quantity <= 0:  {df_neg_qty.count():,}")
print(f"Rows with Price <= 0:     {df_zero_price.count():,}")
print(f"Rows with Price < 0:      {df_neg_price.count():,}")

print("\nSample negative quantities:")
display(df_neg_qty.limit(5))

In [0]:
#date range and country distribution


from pyspark.sql.functions import min, max

#date range
df_raw.select(
    min("InvoiceDate").alias("earliest_date"),
    max("InvoiceDate").alias("latest_date")
).show()

#country distribution
print("\nTop 10 countries by transaction count:")
display(
    df_raw.groupBy("Country")
          .count()
          .orderBy(col("count").desc())
          .limit(10)
)

In [0]:
#duplicate invoice line check

total      = df_raw.count()
distinct   = df_raw.distinct().count()
duplicates = total - distinct

print(f"Total rows:    {total:,}")
print(f"Distinct rows: {distinct:,}")
print(f"Duplicates:    {duplicates:,}")

In [0]:
#data Quality Summary

print("-" * 50)
print("RETAILPULSE — DATA QUALITY REPORT")
print("-" * 50)
print(f"Total raw rows:         {total_rows:,}")
print(f"Cancelled orders:       {df_cancelled.count():,}")
print(f"Negative quantities:    {df_neg_qty.count():,}")
print(f"Zero/negative prices:   {df_zero_price.count():,}")
print(f"Duplicate rows:         {duplicates:,}")
print("-" * 50)